In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import copy
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import pickle
import sys

# Load

In [ ]:
artnet_2024 = **Tabular Data**

In [ ]:
artnet_2024_embed = **Embedding Data **

In [ ]:
year_to_prior = {}
current_prior = set()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Knowledge")
for year in sorted(artnet_2024["workyear from"].unique()):
    year_rows_index = artnet_2024[artnet_2024["workyear from"] == year].index
    year_to_prior[year] = copy.deepcopy(current_prior)  # store snapshot before update
    current_prior |= set(year_rows_index)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Knowledge")

In [ ]:
BASELINE_YEAR=1600

In [ ]:
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Model")
gmms=[]
for year in sorted(artnet_2024["workyear from"].unique()):
    if year < BASELINE_YEAR:
        continue
    prior_knowledge = artnet_2024_embed_reduced[list(year_to_prior.get(year, set()))]
    gmm = GaussianMixture(n_components=20, covariance_type='diag',reg_covar=1e-3) # 20 is a good amount of components
    gmm.fit(prior_knowledge)
    gmms.append(gmm)
    with open(f"ViT_gmm_{year}.pkl", "wb") as f:
        pickle.dump(gmm, f)
    if year%25 ==0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Currently at year {year}")
        
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Model")

In [ ]:
gmms_by_year={}
count=0
for year in sorted(artnet_2024["workyear from"].unique()):
    if year < BASELINE_YEAR:
        continue
    prior_knowledge = artnet_2024_embed_reduced[list(year_to_prior.get(year, set()))]
    gmm = gmms[count]
    count+=1
    gmms_by_year[year] = gmm
    if year%25 ==0:
        print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Currently at year {year}")

In [ ]:
with open(f"gmm_dict.pkl", "wb") as f:
    pickle.dump(gmms_by_year, f)

In [ ]:
sys.getsizeof(gmms_by_year)

In [ ]:
def pointwise_surprise(x, gmm_P):
    x = x.reshape(1, -1) 
    log_prob = gmm_P.score(x)
    return -log_prob  # higher means more surprising

In [ ]:
def compute_surprise(i,year, gmms_by_year,embedding,BASELINE_YEAR):
    if year <= BASELINE_YEAR:  # pre-year baseline
        return i, 0
    # print(f"The total steps are: {playcount}")
    # mark new pairs not in prior knowledge
    gmm = gmms_by_year[year]
    surprise = pointwise_surprise(embedding,gmm)
    return i, surprise

In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
max_workers = 20
BASELINE_YEAR=1600
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    surprise = np.zeros(N)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(compute_surprise, i, 
                    artnet_2024.iloc[i]["workyear from"],
                    gmms_by_year,
                    artnet_2024_embed_reduced[i],
                    BASELINE_YEAR): i
            for i in range(range_start,range_end)
        }
        completed = 0
        for fut in as_completed(futures):
            i,row_surprise = fut.result()
            surprise[i-range_start]  = row_surprise
            completed += 1
            if completed % 10000 == 0:
                print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2024.shape[0]}")
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    np.save(f"{path}\\Creativity_Artnet\\Datasets\\Surprise_2024\\surprise_{range_start}.npy", surprise)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")